```
# Program: DuET v1.1.0
# Author: Sungho Lee, Jae-Won Lee
# Affiliation: MOGAM Institute for Biomedical Research
# Contact: https://github.com/mogam-ai/DuET/issues
# Citation: TBD
```

# DuET Downstream Analysis: Embedding Space

Extract head-input embeddings from the trained DuET model and visualize with PCA/UMAP.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, Normalize
from glob import glob
from os.path import getmtime
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader
import umap.umap_ as umap

from duet.models.module import Module
from duet.configs.config import load_cfgs
from duet.data.datamodule import DataModule


## 1. Load Model and Data

In [ ]:
# MULTITASK DuET v5 (76 cell-type outputs). extract_embeddings = shared
# representation (head-independent), so embedding extraction is identical to single.
exp_dir = "_logs/duet-utr5.500+cds.3000-seed42-split00"
model_ckpt = sorted(glob(f"{exp_dir}/ckpts/*.ckpt"), key=getmtime)[-1]

cfg, dict_cfg = load_cfgs([f"{exp_dir}/config.yaml"], {})
model = Module.load_from_checkpoint(model_ckpt, cfg=cfg, dict_cfg=dict_cfg, strict=False)
model.eval()
model.cuda()
print(f"Loaded: {model_ckpt}")


In [ ]:
# Setup datamodule with full dataset (no kfold split)
scaler_path = f"{exp_dir}/label_scaler.joblib"
scaler_obj = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

# Override kfold to use full data
cfg.datamodule.do_kfold_test = False
datamodule = DataModule(cfg, cfg.dataset.test, scaler_obj=scaler_obj)
datamodule.setup("test")

data_loader = DataLoader(datamodule.test_dataset, batch_size=128, shuffle=False, num_workers=1)
print(f"Full dataset size: {len(datamodule.test_dataset)}")


## 2. Extract Embeddings

In [ ]:
raw_embeddings = []
with torch.no_grad():
    for batch in data_loader:
        batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        emb = model.model.extract_embeddings(batch)
        raw_embeddings.append(emb.cpu().numpy())

feature_vector = np.concatenate(raw_embeddings, axis=0)
print(f"Embedding shape: {feature_vector.shape}")
# Expected: (N, 512) = utr_branch(256) + cds_branch(256)  # DuET v5 (cnn64 + gru96*2 per branch)

scaler = StandardScaler()
scaled_embeddings = scaler.fit_transform(feature_vector)


In [ ]:
# Branch-level norm analysis
utr_dim = model.model.utr_branch.output_dim
cds_dim = model.model.cds_branch.output_dim
print(f"UTR branch dim: {utr_dim}, CDS branch dim: {cds_dim}")

utr_norms = np.linalg.norm(feature_vector[:, :utr_dim], axis=1)
cds_norms = np.linalg.norm(feature_vector[:, utr_dim:utr_dim+cds_dim], axis=1)
print(f"UTR mean norm: {utr_norms.mean():.3f}, CDS mean norm: {cds_norms.mean():.3f}")


## 3. Metadata

In [ ]:
metadata = datamodule.dataset.data[["txID"]].reset_index(drop=True)

metadata_path = "datasets/sequence_features.tsv"
metadata = metadata.merge(pd.read_csv(metadata_path, sep="	", header=0), how="inner", on="txID")
print(f"Metadata shape: {metadata.shape}")


## 4. PCA

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_embeddings)

pca_3d = PCA(n_components=3)
pca_result_3d = pca_3d.fit_transform(scaled_embeddings)

print(f"PCA 2D explained variance: {pca.explained_variance_ratio_} (sum={pca.explained_variance_ratio_.sum():.3f})")


In [ ]:
# Save PCA
pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=metadata.txID).to_csv(f"{exp_dir}/pca_2d.tsv", sep="	")


In [ ]:
# PCA scatter colored by TE label
# multitask: no scalar `label`; use mean over 76 cell-type TEs (NaN-masked)
_tecols = [c for c in datamodule.dataset.data.columns if c.startswith("TE_")]
labels = np.nanmean(datamodule.dataset.data[_tecols].to_numpy(), axis=1)

inferno = cm.get_cmap("inferno")
truncated_inferno = LinearSegmentedColormap.from_list("truncated_inferno", inferno(np.linspace(0, 0.9, 256)))

plt.figure(figsize=(12, 10))
plt.grid(True, alpha=0.3)
scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], s=3, cmap=truncated_inferno,
                      c=labels, vmax=1.8)
plt.colorbar(scatter, label="Log ratio TE")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA - DuET v3 Embeddings (colored by TE)")
plt.tight_layout()
plt.savefig(f"{exp_dir}/pca_2d_te.png", dpi=150)
plt.show()


## 5. UMAP

In [ ]:
# PCA pre-reduction: use 50 components, check explained variance
pca_pre = PCA(n_components=30)
pca_pre_result = pca_pre.fit_transform(scaled_embeddings)
print(f"PCA 30-dim cumulative explained variance: {pca_pre.explained_variance_ratio_.sum():.3f}")

umap_opt = {
    "n_neighbors": 30,
    "min_dist": 0.1,
    "metric": "cosine",
    "n_jobs": 30,
}

reducer = umap.UMAP(**umap_opt)
umap_embeddings = reducer.fit_transform(pca_pre_result)
print(f"UMAP 2D shape: {umap_embeddings.shape}")


In [ ]:
# UMAP scatter colored by TE
plt.figure(figsize=(12, 10))
plt.grid(True, alpha=0.3)
scatter = plt.scatter(umap_embeddings[:, 0], umap_embeddings[:, 1], s=5, cmap=truncated_inferno,
                      c=labels, vmin=-3, vmax=3)
plt.colorbar(scatter, label="log ratio TE")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.title("UMAP - DuET v5 Embeddings (colored by TE)")
plt.tight_layout()
#plt.savefig(f"{exp_dir}/umap_2d_te.png", dpi=150)
plt.show()


In [ ]:
# Save UMAP
pd.DataFrame(umap_embeddings, columns=["UMAP1", "UMAP2"], index=metadata.txID).to_csv(f"{exp_dir}/umap_2d_multitask.tsv", sep="	")

from pprint import pprint
with open(f"{exp_dir}/umap_2d_params.txt", "w") as f:
    pprint(reducer.get_params(), stream=f)


## 8. UMAP colored by cell-type TE (multitask-specific)
Multitask learns a shared representation that predicts 76 cell-type TEs jointly. Color the same UMAP by individual cell-type TEs to reveal cell-type-specific structure the single (pan-celltype) model cannot show. Edit `PANEL_CELLTYPES` to choose which.


In [ ]:
# Per-cell-type TE coloring on the shared-embedding UMAP
_tecols = [c for c in datamodule.dataset.data.columns if c.startswith('TE_')]
_te = datamodule.dataset.data[_tecols].to_numpy()  # (N,76), NaN-masked
_names = [c[3:] for c in _tecols]

# choose cell types to display (default: a diverse subset + pan-celltype reference)
PANEL_CELLTYPES = ['all-celltype', 'a549', 'b-cell', 'hek293', 'hela', 'k562']
PANEL_CELLTYPES = [c for c in PANEL_CELLTYPES if c in _names]
if not PANEL_CELLTYPES:
    # fallback: 6 cell types with the most non-NaN measurements
    _cov = (~np.isnan(_te)).sum(0)
    PANEL_CELLTYPES = [_names[i] for i in np.argsort(_cov)[::-1][:6]]

ncol = 3
nrow = int(np.ceil(len(PANEL_CELLTYPES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 5 * nrow), squeeze=False)
for ax, ct in zip(axes.ravel(), PANEL_CELLTYPES):
    j = _names.index(ct)
    c = _te[:, j]
    valid = ~np.isnan(c)
    ax.scatter(umap_embeddings[~valid, 0], umap_embeddings[~valid, 1],
               s=3, color='lightgrey', alpha=0.3, label='no TE')
    sc = ax.scatter(umap_embeddings[valid, 0], umap_embeddings[valid, 1],
                    s=5, c=c[valid], cmap=truncated_inferno, vmin=-3, vmax=3)
    fig.colorbar(sc, ax=ax, label='log ratio TE', fraction=0.046)
    ax.set_title(f'{ct}  (n={int(valid.sum())})')
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2'); ax.grid(True, alpha=0.3)
for ax in axes.ravel()[len(PANEL_CELLTYPES):]:
    ax.axis('off')
fig.suptitle('Multitask embedding UMAP colored by cell-type TE', y=1.005)
plt.tight_layout()
plt.savefig(f'{exp_dir}/umap_2d_celltype_te.png', dpi=150, bbox_inches='tight')
plt.show()
print('cell types shown:', PANEL_CELLTYPES)


## 6. UMAP 3D

In [ ]:
reducer_3d = umap.UMAP(n_components=3, **umap_opt)
umap_embeddings_3d = reducer_3d.fit_transform(pca_pre_result)
print(f"UMAP 3D shape: {umap_embeddings_3d.shape}")


In [ ]:
import plotly.express as px

df_3d = pd.DataFrame(umap_embeddings_3d, columns=["UMAP1", "UMAP2", "UMAP3"])
df_3d["label"] = labels

fig = px.scatter_3d(df_3d, x="UMAP1", y="UMAP2", z="UMAP3",
                    color="label", range_color=[-3, 3], color_continuous_scale="inferno",
                    title="3D UMAP - DuET v5 Embeddings")
fig.update_traces(marker_size=1.5)
fig.update_layout(autosize=False, width=1000, height=800)
#fig.write_html(f"{exp_dir}/umap_3d.html")
fig.show()


## 7. Per-Branch UMAP (UTR5 vs CDS)


In [ ]:
# Separate branch embeddings and run UMAP independently
utr_dim = model.model.utr_branch.output_dim
cds_dim = model.model.cds_branch.output_dim

emb_utr = feature_vector[:, :utr_dim]
emb_cds = feature_vector[:, utr_dim:utr_dim+cds_dim]

scaled_utr = StandardScaler().fit_transform(emb_utr)
scaled_cds = StandardScaler().fit_transform(emb_cds)

reducer_utr = umap.UMAP(**umap_opt)
umap_utr = reducer_utr.fit_transform(PCA(n_components=30).fit_transform(scaled_utr))

reducer_cds = umap.UMAP(**umap_opt)
umap_cds = reducer_cds.fit_transform(PCA(n_components=30).fit_transform(scaled_cds))

print(f"UTR UMAP: {umap_utr.shape}, CDS UMAP: {umap_cds.shape}")


In [ ]:
# UTR5 branch UMAP
plt.figure(figsize=(12, 10))
plt.grid(True, alpha=0.3)
scatter = plt.scatter(umap_utr[:, 0], umap_utr[:, 1], s=5, cmap=truncated_inferno,
                      c=labels, vmin=-3, vmax=3)
plt.colorbar(scatter, label="log1p(TE)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.title("UMAP - UTR5 Branch Embeddings (colored by TE)")
plt.tight_layout()
plt.savefig(f"{exp_dir}/umap_2d_utr5.png", dpi=150)
plt.show()


In [ ]:
# CDS branch UMAP
plt.figure(figsize=(12, 10))
plt.grid(True, alpha=0.3)
scatter = plt.scatter(umap_cds[:, 0], umap_cds[:, 1], s=5, cmap=truncated_inferno,
                      c=labels, vmin=-3, vmax=3)
plt.colorbar(scatter, label="log1p(TE)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.title("UMAP - CDS Branch Embeddings (colored by TE)")
plt.tight_layout()
plt.savefig(f"{exp_dir}/umap_2d_cds.png", dpi=150)
plt.show()


In [ ]:
# Save per-branch UMAPs
pd.DataFrame(umap_utr, columns=["UMAP1", "UMAP2"], index=metadata.txID).to_csv(f"{exp_dir}/umap_2d_utr5_multitask.tsv", sep="\t")
pd.DataFrame(umap_cds, columns=["UMAP1", "UMAP2"], index=metadata.txID).to_csv(f"{exp_dir}/umap_2d_cds_multitask.tsv", sep="\t")
